# Structural Code Summarization — Analysis

Statistical analysis and visualizations for the paper:
- **RQ1**: Compression metrics
- **RQ2**: Task-based evaluation
- **RQ3**: Cost-effectiveness analysis

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["figure.dpi"] = 150

## RQ1: Compression Metrics

In [ ]:
# Load RQ1 data
df = pd.read_csv("results/rq1_metrics.csv")
print(f"Total files: {len(df)}")
print(f"Projects: {df['project'].nunique()}")
df.head()

In [ ]:
# Overall compression statistics
metrics = ["compression_ratio", "token_reduction_rate", "information_density"]
summary_stats = df[metrics].describe().T
summary_stats["median"] = df[metrics].median()
print("=== Overall Compression Statistics ===")
print(summary_stats[["mean", "median", "std", "min", "max"]].round(4))

In [ ]:
# Per-project statistics
per_project = df.groupby("project").agg(
    files=("file_name", "count"),
    total_orig_tokens=("original_tokens", "sum"),
    total_summ_tokens=("summary_tokens", "sum"),
    mean_cr=("compression_ratio", "mean"),
    median_cr=("compression_ratio", "median"),
    mean_trr=("token_reduction_rate", "mean"),
    mean_id=("information_density", "mean"),
).round(4)
per_project["overall_cr"] = (per_project["total_summ_tokens"] / per_project["total_orig_tokens"]).round(4)
print("=== Per-Project Statistics ===")
per_project

In [ ]:
# Box plot: Compression Ratio by Project Size Category
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, metric, title in zip(
    axes,
    ["compression_ratio", "token_reduction_rate", "information_density"],
    ["Compression Ratio (lower=better)", "Token Reduction Rate (higher=better)", "Information Density"],
):
    sns.boxplot(data=df, x="project_size_category", y=metric, ax=ax, order=["small", "medium", "large"])
    ax.set_title(title)
    ax.set_xlabel("Project Size")

plt.tight_layout()
plt.savefig("results/rq1_boxplots.png", bbox_inches="tight")
plt.show()

In [ ]:
# Scatter: LOC vs Compression Ratio
fig, ax = plt.subplots(figsize=(10, 6))
scatter = ax.scatter(
    df["original_lines"],
    df["compression_ratio"],
    c=df["language"].map({"kotlin": "#7F52FF", "java": "#F89820"}),
    alpha=0.5,
    s=20,
)
ax.set_xlabel("Original Lines of Code")
ax.set_ylabel("Compression Ratio")
ax.set_title("File Size vs Compression Ratio")
ax.legend(
    handles=[
        plt.Line2D([0], [0], marker="o", color="w", markerfacecolor="#7F52FF", label="Kotlin"),
        plt.Line2D([0], [0], marker="o", color="w", markerfacecolor="#F89820", label="Java"),
    ]
)
plt.savefig("results/rq1_scatter_loc_cr.png", bbox_inches="tight")
plt.show()

In [ ]:
# Stratify by language
print("=== By Language ===")
lang_stats = df.groupby("language")[metrics].agg(["mean", "median", "std"]).round(4)
print(lang_stats)

## RQ2: Task-Based Evaluation

In [ ]:
import json

# Load RQ2 data
with open("results/rq2_responses.json") as f:
    rq2_raw = json.load(f)

rq2 = pd.DataFrame(rq2_raw)
print(f"Total responses: {len(rq2)}")
print(f"Conditions: {rq2['condition'].unique()}")
rq2.head()

In [ ]:
# Load evaluation scores (manually scored)
# Expected format: CSV with columns: task_id, condition, accuracy_score, completeness_score
try:
    scores = pd.read_csv("results/rq2_scores.csv")
    rq2_scored = rq2.merge(scores, on=["task_id", "condition"])
    print(f"Scored responses: {len(rq2_scored)}")
except FileNotFoundError:
    print("rq2_scores.csv not found — run scoring first")
    rq2_scored = None

In [ ]:
# Token usage comparison across conditions
if "input_tokens" in rq2.columns:
    token_comparison = rq2.groupby("condition").agg(
        mean_input_tokens=("input_tokens", "mean"),
        total_input_tokens=("input_tokens", "sum"),
        mean_output_tokens=("output_tokens", "mean"),
        mean_latency=("latency_seconds", "mean"),
    ).round(2)
    print("=== Token Usage by Condition ===")
    print(token_comparison)

In [ ]:
# Paired comparison across conditions: Wilcoxon signed-rank + Cliff's delta
#
# Pairs are formed with an inner merge on task_id, not by sorting both frames
# and trusting them to line up. A task that failed in one condition — the Full
# context does not always fit inside the budget — drops out of that one pair
# instead of cancelling the whole test.

PAIRS = [("full", "summary"), ("full", "hybrid"), ("summary", "hybrid")]
SCORE_COLS = ["accuracy_score", "completeness_score"]


def cliffs_delta(x, y):
    """Dominance statistic. Thresholds from Romano et al. (2006)."""
    more = sum(1 for xi in x for yi in y if xi > yi)
    less = sum(1 for xi in x for yi in y if xi < yi)
    return (more - less) / (len(x) * len(y))


def interpret_delta(d):
    a = abs(d)
    if a < 0.147:
        return "negligible"
    if a < 0.330:
        return "small"
    if a < 0.474:
        return "medium"
    return "large"


def compare(scored, cond_a, cond_b, score_col, group="overall"):
    """One paired comparison. Returns None when no task ran in both conditions."""
    a = scored[scored["condition"] == cond_a][["task_id", score_col]]
    b = scored[scored["condition"] == cond_b][["task_id", score_col]]
    pair = a.merge(b, on="task_id", suffixes=("_a", "_b"))
    if pair.empty:
        return None

    xa = pair[f"{score_col}_a"].to_numpy()
    xb = pair[f"{score_col}_b"].to_numpy()

    if (xa == xb).all():
        # Wilcoxon is undefined when every pair ties; report it rather than raise.
        stat, p = float("nan"), 1.0
    else:
        stat, p = stats.wilcoxon(xa, xb)

    d = cliffs_delta(xa, xb)
    return {
        "group": group,
        "metric": score_col.replace("_score", ""),
        "pair": f"{cond_a} vs {cond_b}",
        "n": len(pair),
        "mean_a": round(xa.mean(), 2),
        "mean_b": round(xb.mean(), 2),
        "W": stat,
        "p_raw": round(p, 4),
        "delta": round(d, 3),
        "magnitude": interpret_delta(d),
    }


def holm(rows, alpha=0.05):
    """Holm-Bonferroni within one family of comparisons."""
    order = sorted(range(len(rows)), key=lambda i: rows[i]["p_raw"])
    m = len(rows)
    running = 0.0
    for rank, i in enumerate(order):
        adj = min(1.0, (m - rank) * rows[i]["p_raw"])
        running = max(running, adj)  # keep adjusted p-values monotone
        rows[i]["p_holm"] = round(running, 4)
        rows[i]["significant"] = bool(running < alpha)
    return rows


def run_comparisons(scored, group="overall"):
    """All three condition pairs, corrected within each metric separately.

    The family is one metric's three pairwise comparisons: correcting accuracy
    and completeness together would penalise each for the other's tests.
    """
    out = []
    for col in SCORE_COLS:
        family = []
        for cond_a, cond_b in PAIRS:
            row = compare(scored, cond_a, cond_b, col, group)
            if row is not None:
                family.append(row)
        out.extend(holm(family))
    return out


if rq2_scored is not None:
    tiers_present = (
        sorted(rq2_scored["model"].dropna().unique())
        if "model" in rq2_scored.columns
        else []
    )
    if len(tiers_present) > 1:
        rows = []
        for tier in tiers_present:
            rows.extend(
                run_comparisons(rq2_scored[rq2_scored["model"] == tier], group=tier)
            )
        overall_tests = pd.DataFrame(rows)
        print("=== Per model: Wilcoxon signed-rank, Holm-corrected within each metric ===")
    else:
        overall_tests = pd.DataFrame(run_comparisons(rq2_scored))
        print("=== Overall: Wilcoxon signed-rank, Holm-corrected within each metric ===")
    print(overall_tests.to_string(index=False))
else:
    overall_tests = None
    tiers_present = []
    print("No scores loaded — run scoring first")


In [ ]:
# Score comparison plot
if rq2_scored is not None:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    for ax, score_col, title in zip(
        axes,
        ["accuracy_score", "completeness_score"],
        ["Accuracy Score", "Completeness Score"],
    ):
        sns.boxplot(
            data=rq2_scored,
            x="condition",
            y=score_col,
            order=["full", "summary", "hybrid"],
            ax=ax,
        )
        ax.set_title(title)
        ax.set_ylim(0.5, 5.5)

    plt.tight_layout()
    plt.savefig("results/rq2_scores_boxplot.png", bbox_inches="tight")
    plt.show()

In [ ]:
# Per-category breakdown: means, then the same paired tests within each category
#
# Finding 2 makes claims per category ("navigation shows no significant
# difference", "understanding degrades"). Means alone cannot support those.

if rq2_scored is not None:
    cat_means = (
        rq2_scored.groupby(["category", "condition"])
        .agg(
            n=("accuracy_score", "size"),
            mean_accuracy=("accuracy_score", "mean"),
            mean_completeness=("completeness_score", "mean"),
        )
        .round(2)
    )
    print("=== Mean scores by category and condition ===")
    print(cat_means)

    rows = []
    for category in sorted(rq2_scored["category"].unique()):
        subset = rq2_scored[rq2_scored["category"] == category]
        rows.extend(run_comparisons(subset, group=category))

    category_tests = pd.DataFrame(rows)
    print("\n=== Per-category tests, Holm-corrected within each category and metric ===")
    print(category_tests.to_string(index=False))

    # A non-significant result is not evidence of equivalence. If the paper wants
    # to claim Summary matches Full on navigation, that needs an equivalence test
    # (TOST) with a margin fixed in advance — not p > 0.05.
    ns = category_tests[~category_tests["significant"]]
    if len(ns):
        print(
            f"\n{len(ns)} comparison(s) not significant. Report these as "
            "'no difference detected', not as 'equivalent'."
        )
else:
    category_tests = None


## RQ2b: How the gap behaves across model capability tiers


In [ ]:
# Does the Full - Summary gap move with model capability?
#
# Three tiers is three points, and "capability" here is an ordinal label, not a
# measured quantity. What follows describes a pattern across tiers. It is not
# evidence that capability is what orders them.
#
# Page's trend test is used rather than Jonckheere-Terpstra because the same
# tasks run on every tier: the observations are blocked, not independent.

TIER_ORDER = ["claude-haiku-4-5", "claude-sonnet-5", "claude-opus-5"]


def gap_table(scored, score_col, cond_a="full", cond_b="summary"):
    """Per-task (cond_a - cond_b) gap, one column per tier, tiers in order."""
    cols = ["task_id", "model", score_col]
    a = scored[scored["condition"] == cond_a][cols]
    b = scored[scored["condition"] == cond_b][cols]
    pair = a.merge(b, on=["task_id", "model"], suffixes=("_a", "_b"))
    pair["gap"] = pair[f"{score_col}_a"] - pair[f"{score_col}_b"]
    wide = pair.pivot(index="task_id", columns="model", values="gap")
    tiers = [m for m in TIER_ORDER if m in wide.columns]
    return wide.reindex(columns=tiers).dropna(), tiers


if rq2_scored is not None and "model" in rq2_scored.columns:
    for score_col in SCORE_COLS:
        wide, tiers = gap_table(rq2_scored, score_col)
        label = score_col.replace("_score", "")
        if len(tiers) < 2 or wide.empty:
            print(f"{label}: fewer than two tiers with paired tasks — skipping")
            continue

        print(f"\n=== {label}: Full minus Summary, by tier ===")
        print(
            pd.DataFrame(
                {
                    "tier": tiers,
                    "n_tasks": len(wide),
                    "mean_gap": [round(wide[t].mean(), 3) for t in tiers],
                    "median_gap": [round(wide[t].median(), 3) for t in tiers],
                    "sd": [round(wide[t].std(ddof=1), 3) for t in tiers],
                }
            ).to_string(index=False)
        )

        if len(tiers) < 3:
            print("Page's trend test needs three tiers; run the remaining sweep.")
            continue

        # Both directions, so the data picks the story rather than the hypothesis.
        up = stats.page_trend_test(wide[tiers].to_numpy())
        down = stats.page_trend_test(wide[tiers[::-1]].to_numpy())
        print(f"Page's L, gap widens with tier:   L={up.statistic:.1f}, p={up.pvalue:.4f}")
        print(f"Page's L, gap narrows with tier:  L={down.statistic:.1f}, p={down.pvalue:.4f}")
        print(
            "A significant L says the gaps are ordered across these three tiers. "
            "It does not say capability caused the ordering, and three points "
            "cannot separate capability from anything else that differs between "
            "these models."
        )

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    for ax, score_col in zip(axes, SCORE_COLS):
        wide, tiers = gap_table(rq2_scored, score_col)
        if wide.empty:
            continue
        sns.boxplot(data=wide[tiers], ax=ax)
        ax.axhline(0, color="grey", lw=1, ls="--")
        ax.set_title(f"Full - Summary gap: {score_col.replace('_score', '')}")
        ax.set_ylabel("score difference")
        ax.set_xticks(range(len(tiers)))
        ax.set_xticklabels([t.replace("claude-", "") for t in tiers], rotation=15)
    plt.tight_layout()
    plt.savefig("results/rq2b_gap_by_tier.png", bbox_inches="tight")
    plt.show()
else:
    print("No per-model scores available yet")


## RQ3: Cost-Effectiveness Analysis

In [ ]:
# Claude list pricing, USD per token. Retrieved 2026-09-20 — quote this date
# wherever these figures reach the paper; per-token prices fall over time and an
# undated cost table cannot be checked by a reader later.
PRICING_RETRIEVED = "2026-09-20"
PRICING = {
    "claude-haiku-4-5": {"input": 1.0 / 1_000_000, "output": 5.0 / 1_000_000},
    "claude-sonnet-5": {"input": 2.0 / 1_000_000, "output": 10.0 / 1_000_000},
    "claude-opus-5": {"input": 5.0 / 1_000_000, "output": 25.0 / 1_000_000},
}

model = "claude-sonnet-5"  # adjust as needed
price = PRICING[model]

if "input_tokens" in rq2.columns:
    cost_analysis = rq2.groupby("condition").agg(
        total_input=("input_tokens", "sum"),
        total_output=("output_tokens", "sum"),
    )
    cost_analysis["input_cost"] = cost_analysis["total_input"] * price["input"]
    cost_analysis["output_cost"] = cost_analysis["total_output"] * price["output"]
    cost_analysis["total_cost"] = cost_analysis["input_cost"] + cost_analysis["output_cost"]

    print("=== Cost Analysis ===")
    print(cost_analysis.round(4))

    if "full" in cost_analysis.index and "summary" in cost_analysis.index:
        savings = cost_analysis.loc["full", "total_cost"] - cost_analysis.loc["summary", "total_cost"]
        pct = savings / cost_analysis.loc["full", "total_cost"] * 100
        print(f"\nSavings (summary vs full): ${savings:.4f} ({pct:.1f}%)")

In [ ]:
# Projection: daily developer usage
if "input_tokens" in rq2.columns:
    full_data = rq2[rq2["condition"] == "full"]
    summ_data = rq2[rq2["condition"] == "summary"]

    if len(full_data) > 0 and len(summ_data) > 0:
        avg_full_input = full_data["input_tokens"].mean()
        avg_summ_input = summ_data["input_tokens"].mean()

        reads_per_day = [20, 35, 50]  # conservative, typical, heavy
        print("=== Daily Cost Projection (Sonnet) ===")
        for n in reads_per_day:
            daily_full = n * avg_full_input * price["input"]
            daily_summ = n * avg_summ_input * price["input"]
            monthly_savings = (daily_full - daily_summ) * 22  # work days
            print(f"  {n} reads/day: ${daily_full:.2f} (full) vs ${daily_summ:.2f} (summary) — saves ${monthly_savings:.2f}/month")

In [ ]:
# Intra-rater agreement (Cohen's kappa on re-evaluated subset)
try:
    reeval = pd.read_csv("results/rq2_reeval_scores.csv")
    scores_orig = pd.read_csv("results/rq2_scores.csv")

    merged = reeval.merge(scores_orig, on=["task_id", "condition"], suffixes=("_reeval", "_orig"))

    from sklearn.metrics import cohen_kappa_score

    kappa_acc = cohen_kappa_score(
        merged["accuracy_score_orig"], merged["accuracy_score_reeval"], weights="quadratic"
    )
    kappa_comp = cohen_kappa_score(
        merged["completeness_score_orig"], merged["completeness_score_reeval"], weights="quadratic"
    )
    print(f"Intra-rater kappa (accuracy): {kappa_acc:.4f}")
    print(f"Intra-rater kappa (completeness): {kappa_comp:.4f}")
except FileNotFoundError:
    print("Re-evaluation scores not yet available")
except ImportError:
    print("Install scikit-learn for Cohen's kappa: uv pip install scikit-learn")

## Summary Table (for paper)

In [ ]:
# Generate LaTeX table for RQ1
if 'per_project' in dir():
    latex = per_project[["files", "total_orig_tokens", "total_summ_tokens", "overall_cr", "mean_trr"]].to_latex(
        float_format="%.4f",
        caption="Compression metrics per project",
        label="tab:rq1",
    )
    print(latex)